# Notebook 00 — Course Setup

## Learning Objectives
- Verify that all dependencies (PyTorch, NumPy, Matplotlib) are correctly installed.
- Select the best available compute device (CUDA > MPS > CPU) automatically.
- Set a global random seed for reproducibility across all experiments.
- Understand the repo layout: `src/`, `data/`, `runs/`, `notebooks/`.
- Save environment metadata so you can reproduce results later.

## Big Picture
Before diving into attention mechanisms, we need a clean working environment.
This notebook walks through the one-time setup every Transformer course participant should run first.
All outputs (plots, CSVs, session reports) are saved under `runs/setup/`.

```
repo root/
├── src/          ← all model code lives here
├── data/tiny/    ← tiny datasets for quick experiments
├── runs/         ← all generated outputs (never committed)
├── notebooks/    ← you are here
└── scripts/      ← standalone training scripts
```

## 1. Imports and Path Setup

In [1]:
import sys
import platform
from pathlib import Path

# Add the repo root to sys.path so that `src` is importable
sys.path.insert(0, '..')

import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')  # non-interactive backend — works everywhere
import matplotlib.pyplot as plt

print(f"Python    : {sys.version.split()[0]}")
print(f"PyTorch   : {torch.__version__}")
print(f"NumPy     : {np.__version__}")
print(f"Matplotlib: {matplotlib.__version__}")
print(f"Platform  : {platform.system()} {platform.machine()}")

Python    : 3.13.12
PyTorch   : 2.11.0+cu130
NumPy     : 2.4.4
Matplotlib: 3.10.9
Platform  : Linux x86_64


## 2. Import src Utilities

In [2]:
from src.utils.device import get_best_device, print_device_info
from src.utils.seed import seed_everything
from src.utils.cleanup import prepare_run_dir
from src.utils.reporting import save_metrics_csv, save_markdown_report
from src.utils.paths import RUNS_SETUP_DIR

print("All src imports successful.")

All src imports successful.


## 3. Prepare Run Directory

In [3]:
# Create runs/setup/ and clean any stale outputs from previous runs
prepare_run_dir(RUNS_SETUP_DIR, clean=True)
print(f"Run directory: {RUNS_SETUP_DIR}")

Run directory: /home/arash/PycharmProjects/Transformer/transformers-from-scratch/runs/setup


## 4. Seed Everything

In [4]:
# Seed Python, NumPy, and PyTorch for reproducible results
seed_everything(42)

[seed] Random seed set to 42.


## 5. Device Selection

In [5]:
# Automatically select the best available device
# Priority: CUDA > MPS (Apple Silicon) > CPU
device = get_best_device()
print_device_info(device)
print(f"\nUsing device: {device}")

Platform  : Linux x86_64
Python    : 3.13.12
PyTorch   : 2.11.0+cu130
Device    : cuda
GPU       : NVIDIA GeForce RTX 5080
VRAM      : 16.6 GB

Using device: cuda


## 6. Dataset Section

This setup notebook does not use a real dataset.
We generate a simple sine wave to verify that NumPy + Matplotlib work correctly.

In [6]:
# Generate a simple sine wave as a sanity-check dataset
t = np.linspace(0, 2 * np.pi, 200)
y = np.sin(t)

print(f"Time axis shape : {t.shape}")
print(f"Signal shape    : {y.shape}")
print(f"Signal min/max  : {y.min():.3f} / {y.max():.3f}")

Time axis shape : (200,)
Signal shape    : (200,)
Signal min/max  : -1.000 / 1.000


## 7. Architecture (Text Diagram)

The Transformer family of models all share a common backbone:

```
Input Tokens (integers)
        │
        ▼
  Token Embedding          # integer → dense vector
        │
        ▼
Positional Encoding        # add position information
        │
        ▼
  ┌─────────────────┐
  │  Encoder Block  │  ×N  # self-attention + FFN
  └─────────────────┘
        │
        ▼
  Pooled Representation
        │
        ▼
  Task Head (classify / translate / forecast)
```

This course builds each component from scratch, starting with raw attention math.

## 8. Theory: Why PyTorch?

PyTorch is the de-facto research framework for Transformers. Key reasons:
- **Dynamic computation graph**: easy to debug with standard Python tools.
- **torch.Tensor**: GPU-accelerated N-dimensional arrays with autograd.
- **nn.Module**: composable building blocks for neural networks.
- **Ecosystem**: HuggingFace, Lightning, and most research code use PyTorch.

## 9. Implementation: Tensor Basics

In [7]:
# Demonstrate the key tensor shapes that appear throughout the course
BATCH_SIZE = 2
SEQ_LEN    = 5
D_MODEL    = 8

# A typical input to any Transformer: [batch, seq_len, d_model]
x = torch.randn(BATCH_SIZE, SEQ_LEN, D_MODEL)
print(f"Input tensor shape : {x.shape}  (batch, seq_len, d_model)")

# Move to device
x = x.to(device)
print(f"Device             : {x.device}")

# Basic statistics
print(f"Mean               : {x.mean().item():.4f}")
print(f"Std                : {x.std().item():.4f}")

Input tensor shape : torch.Size([2, 5, 8])  (batch, seq_len, d_model)
Device             : cuda:0
Mean               : 0.0666
Std                : 1.0539


## 10. Sanity Checks

In [8]:
# Verify that torch is working correctly
assert torch.__version__ >= '2', f"Need PyTorch >= 2.0, got {torch.__version__}"

# Verify seed reproducibility
seed_everything(42)
a = torch.randn(3)
seed_everything(42)
b = torch.randn(3)
assert torch.allclose(a, b), "Seed not reproducing same values!"
print("Seed reproducibility check passed.")

# Verify device works
test = torch.ones(2, 2).to(device)
assert test.sum().item() == 4.0
print(f"Device tensor test passed. Device: {test.device}")

print("All sanity checks passed!")

[seed] Random seed set to 42.
[seed] Random seed set to 42.
Seed reproducibility check passed.
Device tensor test passed. Device: cuda:0
All sanity checks passed!


## 11. Visualization: Sine Wave Plot

In [9]:
# Plot and save a simple sine wave as a matplotlib smoke-test
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(t, y, color='steelblue', linewidth=2, label='sin(t)')
ax.set_title('Sine Wave — Setup Smoke Test', fontsize=14)
ax.set_xlabel('t (radians)')
ax.set_ylabel('sin(t)')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()

plot_path = RUNS_SETUP_DIR / 'test_plot.png'
fig.savefig(plot_path, dpi=100)
plt.close(fig)
print(f"Plot saved to: {plot_path}")

Plot saved to: /home/arash/PycharmProjects/Transformer/transformers-from-scratch/runs/setup/test_plot.png


## 12. Save Outputs

In [10]:
# Save device and environment info to CSV
try:
    import torchvision
    tv_version = torchvision.__version__
except ImportError:
    tv_version = 'not installed'

device_info = {
    'python_version': sys.version.split()[0],
    'pytorch_version': torch.__version__,
    'torchvision_version': tv_version,
    'platform': platform.system(),
    'architecture': platform.machine(),
    'device': str(device),
    'cuda_available': str(torch.cuda.is_available()),
    'mps_available': str(hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()),
}

csv_path = RUNS_SETUP_DIR / 'device_info.csv'
save_metrics_csv(device_info, csv_path)
print(f"Device info CSV saved to: {csv_path}")

# Print the saved info
for k, v in device_info.items():
    print(f"  {k:<25} {v}")

Device info CSV saved to: /home/arash/PycharmProjects/Transformer/transformers-from-scratch/runs/setup/device_info.csv
  python_version            3.13.12
  pytorch_version           2.11.0+cu130
  torchvision_version       0.26.0+cu130
  platform                  Linux
  architecture              x86_64
  device                    cuda
  cuda_available            True
  mps_available             False


In [11]:
# Save a session report in Markdown format
report_path = RUNS_SETUP_DIR / 'session_report.md'
save_markdown_report(
    title='Course Setup',
    summary='Environment verified: Python, PyTorch, NumPy, Matplotlib all functional. '
            'Device selection and seed utilities confirmed.',
    metrics=device_info,
    figures=['test_plot.png'],
    tables=['device_info.csv'],
    output_path=report_path,
    device=str(device),
)
print(f"Session report saved to: {report_path}")

Session report saved to: /home/arash/PycharmProjects/Transformer/transformers-from-scratch/runs/setup/session_report.md


## 13. Failure Cases

Common setup problems and how to fix them:

| Problem | Likely Cause | Fix |
|---------|-------------|-----|
| `ModuleNotFoundError: src` | `sys.path` not set | Make sure `sys.path.insert(0, '..')` runs first |
| `CUDA out of memory` | GPU too small | Reduce batch size or use CPU |
| Plots not showing | Wrong backend | Use `matplotlib.use('Agg')` and `fig.savefig()` |
| Results not reproducible | Seed not set | Call `seed_everything(42)` before any random ops |

## 14. Exercises

1. **Version check**: Modify the assert to also check that NumPy version is >= 1.20.
2. **Multi-device test**: Manually test `resolve_device('cpu')` and `resolve_device('cuda')` and observe the fallback warning.
3. **Reproducibility**: Run the cell that generates `torch.randn(3)` with seed=0 vs seed=42. Are the values different?
4. **Cosine wave**: Add a cosine wave to the test plot and re-save the figure.
5. **Path exploration**: Print `REPO_ROOT` from `src.utils.paths` and verify it points to the right directory.

## 15. Key Takeaways

- `get_best_device()` automatically picks CUDA > MPS > CPU — use it everywhere.
- `seed_everything(42)` seeds Python, NumPy, and PyTorch — call it at the top of every notebook.
- `prepare_run_dir(dir, clean=True)` creates and cleans the output folder before each run.
- All outputs (plots, CSVs, reports) go under `runs/` — never commit these files.
- The `src/` package provides reusable building blocks for all models in this course.